# Подбор гиперпараметров CoxPH и RSF (free → jail)

Вход — таблица `free_episodes_features.csv` ноутбука `free-jail.ipynb` (раздел 2): эпизоды свободы, цель (T, E), разбиение `is_test` и признаки набора RFM. Сам `free-jail.ipynb` не меняется.

**Как подбирается.** Только на **обучении**: кросс-валидация по людям, 5 фолдов (все эпизоды человека — в одном фолде). Тест не используется до самого конца. Параметры подбираются **один раз, на наборе RFM**, и потом фиксируются для всех наборов признаков — чтобы сравнение наборов в `free-jail` оставалось честным.

**Критерий выбора — IBS** (меньше — лучше): он учитывает и порядок людей по риску, и точность самой кривой выживания. C-index показан рядом.

| Модель | Сетка |
|---|---|
| CoxPH | `penalizer` ∈ {0.001, 0.01, 0.1, 1} |
| RSF | `min_samples_leaf` ∈ {10, 30, 100} × `max_features` ∈ {"sqrt", 0.5}; 300 деревьев |

В конце лучшие параметры один раз проверяются на тесте рядом с фиксированными из `free-jail` (CoxPH `penalizer=0.1`, RSF `min_samples_leaf=30`, `max_features="sqrt"`).

Вывод — в `данные_этапы_по_обвинениям/free-jail/tuning/`.

## 1. Данные

In [ ]:
from pathlib import Path
import time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sklearn.model_selection import GroupKFold
from survivors import metrics as svm
from survivors.constants import get_y
if not hasattr(np, "trapz"):                                 # survivors вызывает np.trapz, которого нет в numpy 2.x
    np.trapz = np.trapezoid

HERE = Path.cwd()                                            # папка ноутбука: <папка ноутбука>
SRC = HERE / "данные_этапы_по_обвинениям" / "free-jail"
OUT = SRC / "tuning"
OUT.mkdir(exist_ok=True)
SEED = 42

ds = pd.read_csv(SRC / "free_episodes_features.csv", low_memory=False)
NOT_FEAT = ["id", "ep_row", "event_idx", "n_events", "in_custody", "out_custody", "state", "next_reason",
            "T", "E", "is_test", "first_episode"]                     # столбцы таблицы free-jail, которые не признаки
FEAT = [c for c in ds.columns if c not in NOT_FEAT]                   # набор RFM из free-jail (2.3)
assert not ds[FEAT].isna().any().any(), "пропуски в признаках"
tr = ds[~ds["is_test"]].reset_index(drop=True)
te = ds[ds["is_test"]].reset_index(drop=True)
print(f"обучение: {len(tr)} эпизодов, {tr['id'].nunique()} людей; тест: {len(te)} эпизодов; признаков: {len(FEAT)}")
print(FEAT)

## 2. Кросс-валидация

Фолды — `GroupKFold` по `id`: человек целиком попадает в один фолд. Для каждого фолда сетка времени — 200 точек от 0 до 95 %-квантиля T обучающей части фолда, как в `free-jail`. Метрики — те же функции, что в `free-jail` (раздел 4).

Медианы, которыми в `free-jail` заполнены пропуски давности ареста, посчитаны по всему обучению, включая проверочную часть фолда. На подбор это почти не влияет: пропуск отмечен флагом, а для Кокса значение заполнения при флаге не важно.

In [ ]:
FOLDS = list(GroupKFold(n_splits=5).split(tr, groups=tr["id"]))

def to_grid(times, surv, grid):
    """ступенчатая функция выживания -> значения на сетке; до первой точки S = 1"""
    idx = np.searchsorted(times, grid, side="right") - 1
    return np.where(idx >= 0, surv[..., np.clip(idx, 0, None)], 1.0)

def score(S, a, b, grid):
    """C-index (по площади под S) и IBS на части b; обучение - a"""
    S = np.clip(S.astype(np.float64), 1e-12, 1.0)
    y_a = get_y(time=a["T"].values, cens=a["E"].values.astype(bool))
    y_b = get_y(time=b["T"].values, cens=b["E"].values.astype(bool))
    return {"C-index": concordance_index(b["T"], np.trapz(S, grid, axis=1), b["E"]),
            "IBS": float(svm.ibs_remain(y_a, y_b, S, grid, axis=-1))}

def fit_predict(model, params, a, b, grid):
    if model == "CoxPH":
        m = CoxPHFitter(**params).fit(a[FEAT + ["T", "E"]], duration_col="T", event_col="E")
        return m.predict_survival_function(b[FEAT], times=grid).values.T
    m = RandomSurvivalForest(n_estimators=300, n_jobs=-1, random_state=SEED, **params)
    m.fit(a[FEAT].values, Surv.from_arrays(a["E"].astype(bool), a["T"]))
    return to_grid(m.unique_times_, m.predict_survival_function(b[FEAT].values, return_array=True), grid)

GRID_PARAMS = ([("CoxPH", {"penalizer": p}) for p in (0.001, 0.01, 0.1, 1.0)]
               + [("RSF", {"min_samples_leaf": l, "max_features": f}) for l in (10, 30, 100) for f in ("sqrt", 0.5)])

In [ ]:
rows = []
for model, params in GRID_PARAMS:
    t1 = time.time()
    for k, (ia, ib) in enumerate(FOLDS):
        a, b = tr.iloc[ia], tr.iloc[ib]
        grid = np.linspace(0, np.quantile(a["T"], 0.95), 200)
        rows.append({"модель": model, "параметры": str(params), "фолд": k, **score(fit_predict(model, params, a, b, grid), a, b, grid)})
    print(f"{model} {params}: {time.time() - t1:.0f} с")
cv = pd.DataFrame(rows)
cv.to_csv(OUT / "cv_folds.csv", index=False, encoding="utf-8-sig")

## 3. Результаты кросс-валидации

In [ ]:
res = (cv.groupby(["модель", "параметры"])
         .agg(IBS=("IBS", "mean"), sd_IBS=("IBS", "std"), C_index=("C-index", "mean"), sd_C=("C-index", "std"))
         .round(4).sort_values(["модель", "IBS"]))
res.to_csv(OUT / "cv_summary.csv", encoding="utf-8-sig")
display(res)

BEST = {m: eval(res.loc[m].index[0]) for m in ("CoxPH", "RSF")}        # минимальный средний IBS
FIXED = {"CoxPH": {"penalizer": 0.1}, "RSF": {"min_samples_leaf": 30, "max_features": "sqrt"}}   # как в free-jail
for m in BEST:
    gap = res.loc[(m, str(FIXED[m])), "IBS"] - res.loc[(m, str(BEST[m])), "IBS"]
    print(f"{m}: лучшие {BEST[m]}; фиксированные {FIXED[m]} хуже по IBS на {gap:.4f} "
          f"(sd по фолдам ~{res.loc[m, 'sd_IBS'].mean():.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, m in zip(axes, ("CoxPH", "RSF")):
    r = res.loc[m].reset_index()
    ax.errorbar(range(len(r)), r["IBS"], yerr=r["sd_IBS"], fmt="o", capsize=3)
    ax.set_xticks(range(len(r))); ax.set_xticklabels(r["параметры"], rotation=30, ha="right", fontsize=8)
    ax.set_title(f"{m}: IBS по 5 фолдам (меньше - лучше)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "cv_ibs.png", dpi=150); plt.show()

## 4. Проверка на тесте

Один раз: лучшие параметры кросс-валидации и фиксированные из `free-jail`, обучение на всём обучении, оценка на тесте. Сетка — как в `free-jail` (95 %-квантиль T всего обучения).

In [ ]:
GRID = np.linspace(0, np.quantile(tr["T"], 0.95), 200)
rows = []
for m in ("CoxPH", "RSF"):
    for kind, params in (("фиксированные (free-jail)", FIXED[m]), ("подобранные", BEST[m])):
        rows.append({"модель": m, "параметры": kind, "значения": str(params), **score(fit_predict(m, params, tr, te, GRID), tr, te, GRID)})
test = pd.DataFrame(rows).set_index(["модель", "параметры"]).round(4)
test.to_csv(OUT / "test_check.csv", encoding="utf-8-sig")
test